In [2]:
import numpy as np
import nbimporter
from matplotlib import pyplot as plt
from scipy.optimize import curve_fit
from cdt.data import load_dataset
from scipy.stats import gamma, norm
from sklearn.metrics import roc_auc_score
import os
import random
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from cdt.causality import pairwise
from sklearn.preprocessing import MinMaxScaler
import cepairsimplementation as ce
seedR = random.Random(42)
seedN = np.random.default_rng()
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy.special import psi
np.seterr(divide='raise')  # Convert divide-by-zero warnings into exception
from sklearn.mixture import GaussianMixture
import scipy.stats as st
from extra import gmm_mml
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

No GPU automatically detected. Setting SETTINGS.GPU to 0, and SETTINGS.NJOBS to cpu_count.


In [3]:
#From cdt: entropy calculation
def eval_entropy(x):
    """Evaluate the entropy of the input variable.

    :param x: input variable 1D
    :return: entropy of x
    """
    if x.ndim>1:
        x=x.flatten()
    hx = 0
    sx = sorted(x)
    for i, j in zip(sx[:-1], sx[1:]):
        delta = j-i
        if bool(delta):
            hx += np.log(np.abs(delta))
    hx = hx / (len(x) - 1) + psi(len(x)) - psi(1)
    return hx

def integral_diff(x, y):
    
    
    x=x.flatten()
    y=y.flatten()
    # Reorder x and y in increasing order of x

    sort_idx = np.argsort(x)
    x_sorted = x[sort_idx]
    y_sorted = y[sort_idx]
    
    # Compute x1, x2, y1, and y2
    # x1: all entries except the last one, x2: all entries except the first one
    x1 = x_sorted[:-1]
    x2 = x_sorted[1:]
    y1 = y_sorted[:-1]
    y2 = y_sorted[1:]
    
    # Compute result by looping over the zipped vectors and calling func
    soma=0
    for a, b, c, d in zip(x1, x2, y1, y2):
        if np.abs(a-b)>1e-3 and np.abs(c-d)>1e-3:
            soma=soma+np.log(np.abs((d - c) / (b - a)))
    
    return soma

def integral_approx_estimator(x, y):
    """Integral approximation estimator for causal inference.

    :param x: input variable x 1D
    :param y: input variable y 1D
    :return: Return value of the IGCI model >0 if x->y otherwise if return <0
    """
    return ((integral_diff(x,y) - integral_diff(y,x))/len(x))

In [4]:
def IGCI(d,method="entropy",norm="uniform"):
    x,y=d
    if x.shape[1]>1 or y.shape[1]>1:
        return np.nan
    if norm=="uniform":
        scaler=MinMaxScaler()
    else:
        scaler=StandardScaler()
    x=scaler.fit_transform(x)
    y=scaler.fit_transform(y)

    if method=="entropy":
        result= eval_entropy(x)-eval_entropy(y)
    else:
        result= integral_approx_estimator(x,y)
    #print(result)
    #plt.plot(x,y,'.')
    #plt.show()
    #plt.plot(np.sort(x,axis=0),label="x")
    #plt.plot(np.sort(y,axis=0),label="y")
    #plt.legend()
    #plt.show()
    return result

In [5]:
print("Entropy/Uniform",ce.test_tuebingen(IGCI,method="entropy",norm="uniform"))
#print("Entropy/Gaussian",ce.test_tuebingen(IGCI,method="entropy",norm="gaussian"))
#print("Logabs/Uniform",ce.test_tuebingen(IGCI,method="logabs",norm="uniform"))
#print("Logabs/Gaussian",ce.test_tuebingen(IGCI,method="logabs",norm="gaussian"))

Entropy/Uniform (np.float64(0.7085030424777897), np.float64(0.6532616064772362))
